# 01 — Data Pipeline & Feature Engineering
Loads Tardis BTCUSDT trade data, builds bars, computes all 73 features.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from config import *
from utils import (
    BarBuilder, compute_tofi, compute_tofi_divergence,
    compute_basis_features, compute_funding_features,
    compute_realized_moments, compute_microstructure_features
)

DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

## Load Tardis Trade Data

In [ ]:
# Tardis CSV columns: exchange, symbol, timestamp, local_timestamp, id, side, price, amount
# side = taker side: 'buy' (taker bought = seller was maker) or 'sell' (taker sold = buyer was maker)
# Downloaded via: python download_data.py --api-key YOUR_KEY
# Files land in data/{exchange}/{data_type}/YYYY-MM-DD_SYMBOL.csv.gz

import glob

SPOT_DIR = DATA_DIR / 'binance' / 'trades'
PERP_DIR = DATA_DIR / 'binance-futures' / 'trades'
TICKER_DIR = DATA_DIR / 'binance-futures' / 'derivative_ticker'
LIQ_DIR = DATA_DIR / 'binance-futures' / 'liquidations'


def load_tardis_trades(directory: Path) -> pd.DataFrame:
    """Load all daily Tardis trade CSVs from a directory."""
    files = sorted(directory.glob('*.csv.gz'))
    if not files:
        raise FileNotFoundError(f'No .csv.gz files in {directory}')
    print(f'  Loading {len(files)} files from {directory.name}...')
    dfs = [pd.read_csv(f) for f in files]
    df = pd.concat(dfs, ignore_index=True)
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='us', utc=True)
    # side column already has 'buy'/'sell' from Tardis (taker side)
    df = df[['timestamp', 'price', 'amount', 'side']].sort_values('timestamp')
    return df.reset_index(drop=True)


spot_trades = load_tardis_trades(SPOT_DIR)
perp_trades = load_tardis_trades(PERP_DIR)
print(f'Spot trades: {len(spot_trades):,}  |  Perp trades: {len(perp_trades):,}')
print(f'Date range: {spot_trades["timestamp"].min()} to {spot_trades["timestamp"].max()}')

## Build Bars

In [ ]:
builder = BarBuilder()

# 1-minute bars for features
spot_1m = builder.build_bars(spot_trades, bar_sec=60)
perp_1m = builder.build_bars(perp_trades, bar_sec=60)

# 1-second bars for realized moments
spot_1s = builder.build_bars(spot_trades, bar_sec=1)

# Align perp and spot on same index
common_idx = spot_1m.index.intersection(perp_1m.index)
spot_1m = spot_1m.loc[common_idx]
perp_1m = perp_1m.loc[common_idx]

print(f'1-min bars: {len(spot_1m):,}  |  1-sec bars: {len(spot_1s):,}')

## Compute Directional Features

In [ ]:
# TOFI — 9 variants
spot_tofi = compute_tofi(spot_1m, prefix='spot_')
perp_tofi = compute_tofi(perp_1m, prefix='perp_')
tofi_div = compute_tofi_divergence(spot_tofi, perp_tofi).rename('tofi_perp_spot_divergence')

# Cross-exchange basis — 6 variants
basis = compute_basis_features(perp_1m, spot_1m)

print(f'TOFI features: {spot_tofi.shape[1] + perp_tofi.shape[1] + 1}')
print(f'Basis features: {basis.shape[1]}')

In [ ]:
# Funding rate, OI, mark/index price from derivative_ticker
# Tardis derivative_ticker columns: exchange, symbol, timestamp, local_timestamp,
#   funding_timestamp, funding_rate, predicted_funding_rate, open_interest,
#   last_price, index_price, mark_price

funding_rate = pd.Series(dtype=float, name='funding_rate')
oi = pd.Series(dtype=float, name='open_interest')
mark_price = pd.Series(dtype=float, name='mark_price')
index_price = pd.Series(dtype=float, name='index_price')
liquidations = pd.DataFrame()

if TICKER_DIR.exists() and list(TICKER_DIR.glob('*.csv.gz')):
    print('Loading derivative_ticker...')
    ticker_files = sorted(TICKER_DIR.glob('*.csv.gz'))
    ticker_df = pd.concat([pd.read_csv(f) for f in ticker_files], ignore_index=True)
    ticker_df['timestamp'] = pd.to_datetime(ticker_df['timestamp'], unit='us', utc=True)
    ticker_df = ticker_df.set_index('timestamp').sort_index()

    # Resample to 1-min (last value per minute)
    ticker_1m = ticker_df.resample('1min').last()
    funding_rate = ticker_1m['funding_rate'].reindex(spot_1m.index, method='ffill').fillna(0)
    oi = ticker_1m['open_interest'].reindex(spot_1m.index, method='ffill').fillna(0)
    mark_price = ticker_1m['mark_price'].reindex(spot_1m.index, method='ffill')
    index_price = ticker_1m['index_price'].reindex(spot_1m.index, method='ffill')
    print(f'  Ticker rows: {len(ticker_df):,}')

if LIQ_DIR.exists() and list(LIQ_DIR.glob('*.csv.gz')):
    print('Loading liquidations...')
    liq_files = sorted(LIQ_DIR.glob('*.csv.gz'))
    liquidations = pd.concat([pd.read_csv(f) for f in liq_files], ignore_index=True)
    liquidations['timestamp'] = pd.to_datetime(liquidations['timestamp'], unit='us', utc=True)
    # Resample: total liquidation volume per minute per side
    liquidations = liquidations.set_index('timestamp').sort_index()
    liq_1m = liquidations.groupby([pd.Grouper(freq='1min'), 'side'])['amount'].sum().unstack(fill_value=0)
    liq_1m = liq_1m.reindex(spot_1m.index, fill_value=0)
    print(f'  Liquidation rows: {len(liquidations):,}')

funding_feats = compute_funding_features(
    funding_rate,
    oi,
    liquidations if len(liquidations) > 0 else None,
    perp_tofi.get('perp_tofi_5m'),
    mark_price if len(mark_price) > 0 else None,
    index_price if len(index_price) > 0 else None,
)
print(f'Funding features: {funding_feats.shape[1]}')

In [ ]:
# Realized higher moments — 6 variants
moments = compute_realized_moments(spot_1s)
# Resample to 1-min for joining
moments_1m = moments.resample('1min').last().reindex(spot_1m.index, method='ffill')
print(f'Moment features: {moments_1m.shape[1]}')

In [ ]:
# Microstructure features (HAR-RV, VPIN, Kyle's Lambda, etc.) — ~44 variants
micro = compute_microstructure_features(spot_1m)
print(f'Microstructure features: {micro.shape[1]}')

## Assemble Feature Matrix

In [ ]:
features = pd.concat([
    spot_tofi, perp_tofi, tofi_div,
    basis,
    funding_feats,
    moments_1m,
    micro,
], axis=1)

# Add price columns for downstream use
features['spot_close'] = spot_1m['close']
features['perp_close'] = perp_1m['close']
features['log_return_1m'] = np.log(spot_1m['close'] / spot_1m['close'].shift(1))

# 15-min forward binary target: did price go up?
features['target_up'] = (spot_1m['close'].shift(-15) > spot_1m['close']).astype(float)

print(f'Total features: {features.shape[1] - 3}')  # exclude price cols and target
print(f'Rows: {len(features):,}')

In [ ]:
# Save
features.to_parquet(DATA_DIR / 'features.parquet')
print(f'Saved to {DATA_DIR / "features.parquet"}')

## Validation

In [ ]:
# NaN rates by feature group
feat_cols = [c for c in features.columns if c not in ['spot_close', 'perp_close', 'target_up']]
nan_rates = features[feat_cols].isna().mean().sort_values(ascending=False)
print('Top NaN rates:')
print(nan_rates.head(15).to_string())
print(f'\nFeatures with >50% NaN: {(nan_rates > 0.5).sum()}')
print(f'Features with <5% NaN:  {(nan_rates < 0.05).sum()}')

In [ ]:
# Correlation heatmap of directional features
directional_cols = [c for c in features.columns if any(
    k in c for k in ['tofi', 'basis', 'funding', 'skew']
)]
if len(directional_cols) > 2:
    corr = features[directional_cols].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(directional_cols)))
    ax.set_yticks(range(len(directional_cols)))
    ax.set_xticklabels(directional_cols, rotation=90, fontsize=7)
    ax.set_yticklabels(directional_cols, fontsize=7)
    plt.colorbar(im)
    plt.title('Directional Feature Correlations')
    plt.tight_layout()
    plt.show()